In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                         UNIÃO NACIONAL DOS DADOS                             ║
# ║      Unificação de Schema, Merge Incremental e Geração de Figuras/Tabelas    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : Universidade Federal do Ceará
#  Curso       : Engenharia Elétrica
#  Versão      : 11.0 (Correção de Schema Inconsistente no ParquetWriter)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de união nacional (Step 4) da arquitetura de
#  dados do TCC. Ele une os quatro Parquets consolidados por submercado (SIN —
#  Sistema Interligado Nacional: SUDESTE, SUL, NORDESTE, NORTE), gerados na
#  etapa de consolidação, em uma única base "wide" no formato Brasil, com uma
#  linha por combinação de KEY_ANO/KEY_MES/KEY_DIA/KEY_HORA e colunas de cada
#  região prefixadas por submercado.
#
#  A junção é feita por streaming (chunks) para não explodir o uso de RAM, e o
#  schema final do Parquet é construído previamente (lendo apenas metadados)
#  para garantir consistência de tipos entre todos os chunks gravados.
#
#  Ao final da execução, o script gera:
#    (a) um Parquet único em formato wide com todas as regiões do SIN;
#    (b) figuras (PNG) de completude de dados e distribuição de variáveis;
#    (c) tabelas (CSV) de resumo do pipeline e auditoria temporal.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script não requer entradas manuais do usuário. Os parâmetros de controle
#  estão definidos nas constantes abaixo (seção "CONFIGURAÇÕES"):
#
#    ANOS_FILTRO  : anos considerados na união (padrão: 2021 até o ano atual)
#    DIR_RAIZ     : caminho raiz de armazenamento
#                   (Google Drive: /content/drive/MyDrive/TCC_PLD_Project)
#
#  Arquivos de entrada esperados (gerados pela etapa de consolidação):
#    DIR_RAIZ / "2-DADOS" / "3-DADOS-CONSOLIDADOS" /
#        DADOS_CONSOLIDADOS_<REGIAO>.parquet   (uma por submercado)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Parquet unificado →  DIR_RAIZ / "2-DADOS" / "4-DADOS-UNIAO-GERAL" /
#     DADOS_BRASIL_WIDE.parquet   (compressão Snappy, schema unificado)
#
#  2. Figuras (PNG)     →  DIR_RAIZ / "2-DADOS" / "5-FIGURAS-TCC" /
#     FIG01_nan_por_ano.png                    — % de dados faltantes por ano
#     FIG02_heatmap_nan_regiao_ano.png         — completude por região × ano
#     FIG04_pipeline_por_regiao.png            — registros/features por região
#     FIG05_rosca_variaveis_regiao_fonte.png   — distribuição de variáveis
#
#  3. Tabelas (CSV)      →  mesma pasta de figuras
#     TAB01_resumo_pipeline.csv       — resumo do pipeline por região
#     TAB02_auditoria_temporal.csv    — completude e cobertura por ano
#
#  4. Saída no terminal:
#     • Log de progresso por fase (preparação, schema, merge, auditoria)
#     • Auditoria de data de corte com base nas colunas meteorológicas (INMET)
#     • Relatório final com dimensões, tamanho em disco e tempo total
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CORREÇÕES APLICADAS NESTA VERSÃO (v11)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  BUG (v10): chunks gerados via outer-join podiam produzir colunas do tipo
#  `null` quando uma região não tinha dados para aquelas linhas. O pyarrow
#  inferia `null` no primeiro chunk e depois `float`/`string` em chunks
#  seguintes, causando ValueError de schema inconsistente no ParquetWriter.
#
#  CORREÇÃO: antes de abrir o ParquetWriter, o script agora constrói um
#  SCHEMA UNIFICADO lendo apenas o schema de cada Parquet temporário (sem
#  carregar dados em memória), resolve conflitos null→float e null→string,
#  e força cada chunk para esse schema fixo via cast explícito antes de
#  gravar (ver construir_schema_unificado() e _cast_chunk_para_schema()).
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de DIR_RAIZ.
#     Depende dos quatro Parquets consolidados por submercado gerados na
#     etapa anterior (Step 3 — consolidação).
#
#  2. ESTRATÉGIA DE MEMÓRIA (STREAMING JOIN)
#     Cada região é carregada, otimizada (downcast de tipos) e salva como
#     Parquet temporário isoladamente — apenas uma região por vez ocupa RAM
#     na Fase 1. O merge final (Fase 3) é feito em chunks de chaves
#     temporais (KEYS_TIME), nunca carregando o dataset unificado inteiro
#     de uma vez.
#
#  3. RESOLUÇÃO DE SCHEMA (_resolver_tipo)
#     Ao comparar os tipos de uma mesma coluna vindos de diferentes regiões,
#     o script resolve o tipo mais permissivo: null cede para qualquer tipo
#     real; inteiro cede para float32 (evita perda de casas decimais);
#     qualquer conflito com string cai para string como fallback seguro.
#
#  4. NOMENCLATURA DE COLUNAS
#     Ao carregar cada região, todas as colunas de dados (exceto as chaves
#     temporais) recebem o sufixo "_<REGIAO>" para evitar colisão de nomes
#     no merge final (ex.: PLD_SUDESTE, PLD_NORDESTE). A coluna
#     KEY_SUBMERCADO é descartada, já que a informação de região passa a
#     estar embutida no nome da coluna.
#
#  5. AUDITORIA DE DATA DE CORTE
#     auditar_data_corte_meteo() usa as colunas meteorológicas (INMET) como
#     referência de defasagem, por serem tipicamente as mais atrasadas no
#     pipeline (o INMET não disponibiliza dados do ano corrente em ZIP).
#     A leitura é lazy (apenas as colunas necessárias) para não sobrecarregar
#     a memória.
#
#  6. FIGURAS E TABELAS
#     Todas as visualizações e tabelas leem o Parquet final em subconjuntos
#     (lazy, por coluna/filtro), nunca carregando a base wide completa de
#     uma só vez — importante dado o número elevado de colunas após a união
#     das quatro regiões.
#
#  7. LIMPEZA
#     Os Parquets temporários por região (DIR_TMP) são removidos ao final da
#     execução (e também no início, como precaução contra resíduos de
#     execuções anteriores).
#
#  8. REPRODUTIBILIDADE
#     O relatório final no terminal registra dimensões, tamanho em disco,
#     tempo total de execução e a data de corte identificada via INMET,
#     garantindo rastreabilidade da união realizada.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações numéricas e downcast de tipos
#  pandas            1.5            Manipulação de DataFrames e merges
#  pyarrow           10.0           Schema unificado, leitura/escrita Parquet
#  matplotlib        3.6           Geração das figuras
#  seaborn           0.12           Heatmaps e estilização das figuras
#
#  Instalação rápida (Google Colab / pip):
#      !pip install pyarrow -q
#  (numpy, pandas, matplotlib e seaborn já estão disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS UTILIZADA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TCC_PLD_Project/
#  └── 2-DADOS/
#      ├── 3-DADOS-CONSOLIDADOS/        (entrada desta etapa)
#      │   ├── DADOS_CONSOLIDADOS_SUDESTE.parquet
#      │   ├── DADOS_CONSOLIDADOS_SUL.parquet
#      │   ├── DADOS_CONSOLIDADOS_NORDESTE.parquet
#      │   └── DADOS_CONSOLIDADOS_NORTE.parquet
#      ├── 4-DADOS-UNIAO-GERAL/          (saída desta etapa)
#      │   └── DADOS_BRASIL_WIDE.parquet
#      ├── 5-FIGURAS-TCC/                (figuras e tabelas desta etapa)
#      │   ├── FIG01_nan_por_ano.png
#      │   ├── FIG02_heatmap_nan_regiao_ano.png
#      │   ├── FIG04_pipeline_por_regiao.png
#      │   ├── FIG05_rosca_variaveis_regiao_fonte.png
#      │   ├── TAB01_resumo_pipeline.csv
#      │   └── TAB02_auditoria_temporal.csv
#      └── .tmp_merge/                   (Parquets temporários por região)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install pyarrow -q
#
#  Célula 3 — Executar o script:
#      exec(open('uniao_nacional_dados_tcc_v11.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

try:
    import pyarrow
    import pyarrow.parquet as pq
    import pyarrow.dataset as ds
    import pyarrow.compute as pc
except ImportError:
    import os
    os.system("pip install pyarrow -q")
    import pyarrow
    import pyarrow.parquet as pq
    import pyarrow.dataset as ds
    import pyarrow.compute as pc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import gc
import warnings
import time
from datetime import datetime as dt

warnings.simplefilter(action="ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)


# ==============================================================================
# 🎨 UI
# ==============================================================================

class UI:
    RESET   = "\033[0m"; BOLD    = "\033[1m"; CYAN    = "\033[96m"
    GREEN   = "\033[92m"; YELLOW  = "\033[93m"; RED     = "\033[91m"
    BLUE    = "\033[94m"; MAGENTA = "\033[95m"

    @staticmethod
    def banner():
        print(f"{UI.BLUE}")
        print("╔════════════════════════════════════════════════════════════════════╗")
        print("║   🇧🇷  MOTOR DE UNIFICAÇÃO NACIONAL - SIN (SISTEMA INTEGRADO)       ║")
        print("║   STRATEGY: UNIFIED SCHEMA + PARQUET STREAMING JOIN   v11          ║")
        print("╚════════════════════════════════════════════════════════════════════╝")
        print(f"{UI.RESET}")

    @staticmethod
    def step(msg):
        print(f"\n{UI.CYAN}{'─'*70}{UI.RESET}")
        print(f"{UI.CYAN}➤  {msg}{UI.RESET}")

    @staticmethod
    def info(label, value):
        print(f"   🔹 {label:<30} : {UI.BOLD}{value}{UI.RESET}")

    @staticmethod
    def success(msg):  print(f"   {UI.GREEN}✔  {msg}{UI.RESET}")
    @staticmethod
    def warning(msg):  print(f"   {UI.YELLOW}⚠  {msg}{UI.RESET}")
    @staticmethod
    def error(msg):    print(f"   {UI.RED}✖  {msg}{UI.RESET}")


# ==============================================================================
# ⚙️ CONFIGURAÇÕES DO AMBIENTE
# ==============================================================================

DIR_RAIZ    = r"/content/drive/MyDrive/TCC_PLD_Project"
DIR_ENTRADA = os.path.join(DIR_RAIZ, "2-DADOS/3-DADOS-CONSOLIDADOS")
DIR_SAIDA   = os.path.join(DIR_RAIZ, "2-DADOS/4-DADOS-UNIAO-GERAL")
DIR_FIGURAS = os.path.join(DIR_RAIZ, "2-DADOS/5-FIGURAS-TCC")
DIR_TMP     = os.path.join(DIR_RAIZ, "2-DADOS/.tmp_merge")

REGIOES_ORDEM = ["SUDESTE", "SUL", "NORDESTE", "NORTE"]
KEYS_TIME     = ["KEY_ANO", "KEY_MES", "KEY_DIA", "KEY_HORA"]

COR_REGIAO = {
    "SUDESTE": "#1f77b4", "SUL": "#2ca02c",
    "NORDESTE": "#ff7f0e", "NORTE": "#d62728",
}

ANO_ATUAL   = dt.now().year
ANOS_FILTRO = list(range(2021, ANO_ATUAL + 1))

TCC_STYLE = {
    "font.family": "serif", "axes.titlesize": 13, "axes.labelsize": 11,
    "xtick.labelsize": 9,   "ytick.labelsize": 9,  "legend.fontsize": 9,
    "figure.dpi": 150,      "savefig.dpi": 300,    "savefig.bbox": "tight",
}


# ==============================================================================
# 🛠️ FUNÇÕES UTILITÁRIAS
# ==============================================================================

def otimizar_memoria(df: pd.DataFrame) -> tuple:
    start = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.select_dtypes("float64").columns:
        df[col] = df[col].astype(np.float32)
    for col in df.select_dtypes("int64").columns:
        mn, mx = df[col].min(), df[col].max()
        if mn > np.iinfo(np.int16).min and mx < np.iinfo(np.int16).max:
            df[col] = df[col].astype(np.int16)
        elif mn > np.iinfo(np.int32).min and mx < np.iinfo(np.int32).max:
            df[col] = df[col].astype(np.int32)
    end = df.memory_usage(deep=True).sum() / 1024**2
    return df, start, end


def carregar_regiao(regiao: str) -> tuple:
    path = os.path.join(DIR_ENTRADA, f"DADOS_CONSOLIDADOS_{regiao}.parquet")
    if not os.path.exists(path):
        UI.warning(f"Arquivo não encontrado: {path}")
        return None, 0, 0, 0.0, 0.0
    try:
        filters = [("KEY_ANO", "in", ANOS_FILTRO)] if ANOS_FILTRO else None
        df = pd.read_parquet(path, filters=filters)
        if df.empty:
            UI.warning(f"Sem dados nos anos {ANOS_FILTRO} para {regiao}.")
            return None, 0, 0, 0.0, 0.0
        rows, cols = df.shape
        df, ma, md = otimizar_memoria(df)
        mapa = {
            c: f"{c}_{regiao}"
            for c in df.columns
            if c not in KEYS_TIME and c != "KEY_SUBMERCADO"
        }
        df.rename(columns=mapa, inplace=True)
        if "KEY_SUBMERCADO" in df.columns:
            df.drop(columns=["KEY_SUBMERCADO"], inplace=True)
        return df, rows, cols, ma, md
    except Exception as e:
        UI.error(f"Erro ao ler {regiao}: {e}")
        return None, 0, 0, 0.0, 0.0


def limpar_tmp(dir_tmp: str) -> None:
    if not os.path.exists(dir_tmp):
        return
    for f in os.listdir(dir_tmp):
        if f.endswith(".parquet"):
            try:
                os.remove(os.path.join(dir_tmp, f))
            except Exception:
                pass


# ==============================================================================
# 🔑 CONSTRUÇÃO DO SCHEMA UNIFICADO  ← CORREÇÃO CENTRAL DA v11
# ==============================================================================

def _resolver_tipo(tipos: list) -> pyarrow.DataType:
    """
    Dado uma lista de tipos pyarrow vindos de N datasets para a mesma coluna,
    resolve para o tipo "mais rico":
      null  → qualquer outro tipo vence
      int*  → float32 vence (para não perder valores fracionários de outras regiões)
      large_string / string → string
      conflito float vs string → string (fallback seguro)
    """
    tipos_sem_null = [t for t in tipos if t != pyarrow.null()]
    if not tipos_sem_null:
        return pyarrow.float32()           # coluna totalmente null → float32

    tipo_base = tipos_sem_null[0]
    for t in tipos_sem_null[1:]:
        if t == tipo_base:
            continue
        # int + float → float32
        if pyarrow.types.is_integer(tipo_base) and pyarrow.types.is_floating(t):
            tipo_base = pyarrow.float32()
        elif pyarrow.types.is_floating(tipo_base) and pyarrow.types.is_integer(t):
            pass  # mantém float
        # large_string / string → string
        elif pyarrow.types.is_large_string(t) or pyarrow.types.is_string(t):
            tipo_base = pyarrow.string()
        elif pyarrow.types.is_large_string(tipo_base):
            tipo_base = pyarrow.string()
        else:
            # qualquer outro conflito → float32 se ambos numéricos, senão string
            if pyarrow.types.is_floating(t) or pyarrow.types.is_integer(t):
                tipo_base = pyarrow.float32()
            else:
                tipo_base = pyarrow.string()

    # normaliza large_string → string e int* → float32 para consistência
    if pyarrow.types.is_large_string(tipo_base):
        tipo_base = pyarrow.string()
    if pyarrow.types.is_integer(tipo_base):
        tipo_base = pyarrow.float32()

    return tipo_base


def construir_schema_unificado(paths: list) -> pyarrow.Schema:
    """
    Lê apenas os schemas dos parquets temporários (sem carregar dados),
    constrói um schema único resolvendo conflitos de tipo coluna a coluna.
    """
    UI.step("Construindo schema unificado (sem carregar dados)")

    # Mapeia coluna → lista de tipos encontrados em cada dataset
    col_tipos: dict = {}

    for p in paths:
        s = pq.read_schema(p)
        for i, name in enumerate(s.names):
            col_tipos.setdefault(name, []).append(s.types[i])

    campos = []
    conflitos = 0
    for nome, tipos in col_tipos.items():
        tipo_final = _resolver_tipo(tipos)
        # detecta conflito real (tipos distintos após excluir null)
        tipos_reais = set(str(t) for t in tipos if t != pyarrow.null())
        if len(tipos_reais) > 1:
            conflitos += 1
        campos.append(pyarrow.field(nome, tipo_final))

    schema = pyarrow.schema(campos)
    UI.info("Colunas no schema unificado", len(schema))
    UI.info("Conflitos de tipo resolvidos", conflitos)
    return schema


# ==============================================================================
# 🔀 MERGE INCREMENTAL COM SCHEMA FIXO
# ==============================================================================

def _cast_chunk_para_schema(
    df: pd.DataFrame,
    schema: pyarrow.Schema,
) -> pyarrow.Table:
    """
    Converte um chunk pandas para pyarrow.Table forçando o schema unificado.
    Colunas ausentes no chunk (região sem dados naquelas linhas) são
    preenchidas com null e depois castadas para o tipo correto.
    """
    # Garante que todas as colunas do schema existam no df
    for campo in schema:
        if campo.name not in df.columns:
            df[campo.name] = None

    # Reordena colunas para bater com o schema
    df = df[[campo.name for campo in schema]]

    tabela = pyarrow.Table.from_pandas(df, preserve_index=False, safe=False)

    # Cast coluna a coluna para o tipo definido no schema unificado
    arrays_cast = []
    for i, campo in enumerate(schema):
        arr = tabela.column(i)
        try:
            arrays_cast.append(arr.cast(campo.type, safe=False))
        except Exception:
            # fallback: preenche com null do tipo correto
            arrays_cast.append(
                pyarrow.array([None] * len(arr), type=campo.type)
            )

    return pyarrow.table(
        {campo.name: arrays_cast[i] for i, campo in enumerate(schema)},
        schema=schema,
    )


def merge_parquets_por_chave(
    paths_regioes: list,
    arquivo_saida: str,
    schema_unificado: pyarrow.Schema,
    chunksize: int = 5_000,
) -> dict:
    """
    Merge incremental sem explosão de RAM.
    Usa o schema_unificado fixo para garantir consistência entre chunks.
    """
    os.makedirs(os.path.dirname(arquivo_saida), exist_ok=True)

    UI.step("Merge Incremental por Chave (schema fixo, sem explosão de RAM)")

    datasets = [ds.dataset(p, format="parquet") for p in paths_regioes if os.path.exists(p)]

    if not datasets:
        UI.error("Nenhum dataset disponível para merge.")
        return {}

    # Coleta chaves únicas lendo apenas KEYS_TIME
    UI.info("Coletando chaves únicas", "apenas KEYS_TIME por região...")
    chaves_sets = []
    for d in datasets:
        df_k = d.to_table(columns=KEYS_TIME).to_pandas()
        df_k, _, _ = otimizar_memoria(df_k)
        chaves_sets.append(set(map(tuple, df_k[KEYS_TIME].values)))
        del df_k
        gc.collect()

    todas_chaves = sorted(set().union(*chaves_sets))
    del chaves_sets
    gc.collect()

    total_linhas = len(todas_chaves)
    UI.info("Total de chaves únicas (linhas finais)", f"{total_linhas:,}")

    writer = pq.ParquetWriter(arquivo_saida, schema_unificado, compression="snappy")
    linhas_escritas = 0

    for inicio in range(0, total_linhas, chunksize):
        chunk_chaves = todas_chaves[inicio: inicio + chunksize]

        df_chunk_base = pd.DataFrame(chunk_chaves, columns=KEYS_TIME)
        df_chunk_base, _, _ = otimizar_memoria(df_chunk_base)

        df_merged = df_chunk_base.copy()

        for d in datasets:
            # Lê todas as colunas deste dataset apenas para as linhas do chunk
            df_reg = d.to_table().to_pandas()
            df_reg, _, _ = otimizar_memoria(df_reg)
            df_reg = df_reg.merge(df_chunk_base[KEYS_TIME], on=KEYS_TIME, how="inner")

            df_merged = df_merged.merge(df_reg, on=KEYS_TIME, how="left")
            del df_reg
            gc.collect()

        df_merged.sort_values(by=KEYS_TIME, inplace=True)
        df_merged.reset_index(drop=True, inplace=True)

        # ── CAST PARA SCHEMA FIXO ─────────────────────────────────────────────
        tabela = _cast_chunk_para_schema(df_merged, schema_unificado)
        writer.write_table(tabela)

        linhas_escritas += len(df_merged)
        progresso = min(inicio + chunksize, total_linhas)
        UI.info(
            f"  Chunk {inicio//chunksize + 1}",
            f"{progresso:,}/{total_linhas:,} linhas escritas"
        )

        del df_merged, df_chunk_base, tabela
        gc.collect()

    writer.close()

    final_size = os.path.getsize(arquivo_saida) / 1024**2
    UI.success(f"Parquet final: {arquivo_saida} ({final_size:.1f} MB)")

    return {
        "total_linhas":  linhas_escritas,
        "total_colunas": len(schema_unificado),
        "tamanho_mb":    round(final_size, 2),
    }


# ==============================================================================
# 📡 AUDITORIA DE DATA DE CORTE — leitura lazy
# ==============================================================================

def auditar_data_corte_meteo(arquivo: str) -> dict:
    schema    = pq.read_schema(arquivo)
    cols_meteo = [c for c in schema.names if "METEO" in c.upper()]

    if not cols_meteo:
        UI.warning("Nenhuma coluna METEO encontrada.")
        return {"cols_meteo": [], "ultimo_ano": None, "ultimo_mes": None,
                "ultimo_dia": None, "ultima_hora": None, "ultima_data_fmt": "N/A"}

    UI.info("Colunas METEO identificadas", len(cols_meteo))

    cols_ler = KEYS_TIME + cols_meteo[:5]
    df = pd.read_parquet(arquivo, columns=cols_ler)
    df, _, _ = otimizar_memoria(df)

    mask = df[cols_meteo[:5]].notna().any(axis=1)
    df_f = df.loc[mask, KEYS_TIME].sort_values(by=KEYS_TIME)
    del df, mask
    gc.collect()

    if df_f.empty:
        return {"cols_meteo": cols_meteo, "ultimo_ano": None, "ultimo_mes": None,
                "ultimo_dia": None, "ultima_hora": None, "ultima_data_fmt": "N/A"}

    u = df_f.iloc[-1]
    del df_f
    gc.collect()

    return {
        "cols_meteo":     cols_meteo,
        "ultimo_ano":     int(u["KEY_ANO"]),
        "ultimo_mes":     int(u["KEY_MES"]),
        "ultimo_dia":     int(u["KEY_DIA"]),
        "ultima_hora":    int(u["KEY_HORA"]),
        "ultima_data_fmt": f"{int(u['KEY_DIA']):02d}/{int(u['KEY_MES']):02d}/{int(u['KEY_ANO'])}",
    }


# ==============================================================================
# 📊 VISUALIZAÇÕES — todas leem o parquet final em subsets lazy
# ==============================================================================

def _salvar(fig, nome, dir_figuras):
    os.makedirs(dir_figuras, exist_ok=True)
    p = os.path.join(dir_figuras, nome)
    fig.savefig(p)
    UI.success(f"Figura salva: {p}")


def figura_nan_por_ano(arquivo: str, dir_figuras: str) -> None:
    UI.step("Figura 1 — % Dados Faltantes por Ano")
    schema     = pq.read_schema(arquivo)
    cols_dados = [c for c in schema.names if c not in KEYS_TIME]
    anos       = sorted(pd.read_parquet(arquivo, columns=["KEY_ANO"])["KEY_ANO"].unique())

    pct_nan = {}
    for ano in anos:
        df_ano = pd.read_parquet(arquivo, columns=cols_dados,
                                 filters=[("KEY_ANO", "=", ano)])
        arr = df_ano.values
        # trata strings: converte para float substituindo não-numéricos por NaN
        try:
            arr = arr.astype(float)
        except Exception:
            arr = np.array([[
                float(v) if isinstance(v, (int, float)) and not isinstance(v, bool)
                else np.nan for v in row
            ] for row in arr])
        pct_nan[int(ano)] = float(np.isnan(arr).mean() * 100)
        del df_ano, arr
        gc.collect()

    with plt.rc_context(TCC_STYLE):
        fig, ax = plt.subplots(figsize=(9, 4))
        anos_str = [str(a) for a in pct_nan]
        vals     = list(pct_nan.values())
        cores    = ["#27ae60" if v < 10 else "#f1c40f" if v < 40 else "#c0392b" for v in vals]
        barras   = ax.bar(anos_str, vals, color=cores, edgecolor="white",
                          linewidth=0.6, alpha=0.88)
        for b in barras:
            h = b.get_height()
            ax.text(b.get_x() + b.get_width()/2, h + 0.3,
                    f"{h:.1f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")
        ax.set_title("Figura 1 — Concentração de Dados Faltantes por Ano (SIN)",
                     fontweight="bold")
        ax.set_xlabel("Ano"); ax.set_ylabel("% Média de Valores Ausentes")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
        ax.set_ylim(0, min(max(vals) * 1.25 + 5, 105))
        from matplotlib.patches import Patch
        ax.legend(handles=[
            Patch(facecolor="#27ae60", label="< 10 % (Bom)"),
            Patch(facecolor="#f1c40f", label="10–40 % (Atenção)"),
            Patch(facecolor="#c0392b", label="> 40 % (Crítico)"),
        ], loc="upper right", framealpha=0.85)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        sns.despine(ax=ax)
        plt.tight_layout()
    _salvar(fig, "FIG01_nan_por_ano.png", dir_figuras)
    plt.show(); plt.close(fig)
    del pct_nan; gc.collect()


def figura_nan_por_regiao_ano(arquivo: str, dir_figuras: str) -> None:
    UI.step("Figura 2 — Heatmap NaN por Região × Ano")
    schema   = pq.read_schema(arquivo)
    all_cols = schema.names
    anos     = sorted(pd.read_parquet(arquivo, columns=["KEY_ANO"])["KEY_ANO"].unique())

    registros = []
    for regiao in REGIOES_ORDEM:
        cols_reg = [c for c in all_cols if c.endswith(f"_{regiao}")]
        if not cols_reg:
            continue
        for ano in anos:
            df_b = pd.read_parquet(arquivo, columns=cols_reg,
                                   filters=[("KEY_ANO", "=", ano)])
            pct = float(df_b.isna().mean().mean() * 100)
            registros.append({"Região": regiao, "Ano": int(ano), "NaN (%)": round(pct, 2)})
            del df_b
        gc.collect()

    if not registros:
        UI.warning("Dados insuficientes para Figura 2."); return

    pivot = pd.DataFrame(registros).pivot(index="Região", columns="Ano", values="NaN (%)")
    with plt.rc_context(TCC_STYLE):
        fig, ax = plt.subplots(figsize=(10, 3.5))
        sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
                    linewidths=0.4, linecolor="white", vmin=0, vmax=100, ax=ax,
                    cbar_kws={"label": "% NaN", "shrink": 0.8})
        ax.set_title("Figura 2 — Mapa de Completude dos Dados por Região e Ano (%NaN)",
                     fontweight="bold")
        ax.set_xlabel("Ano"); ax.set_ylabel("")
        plt.tight_layout()
    _salvar(fig, "FIG02_heatmap_nan_regiao_ano.png", dir_figuras)
    plt.show(); plt.close(fig)
    del pivot, registros; gc.collect()


def figura_pipeline_report(report_data: list, dir_figuras: str) -> None:
    UI.step("Figura 4 — Painel de Resultados do Pipeline por Região")
    df_rep = pd.DataFrame(report_data)
    df_rep = df_rep[df_rep["Status"] == "MERGED"].copy()
    if df_rep.empty:
        UI.warning("Sem dados para Figura 4."); return

    with plt.rc_context(TCC_STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        cores = [COR_REGIAO.get(r, "#7f7f7f") for r in df_rep["Regiao"]]

        axes[0].barh(df_rep["Regiao"], df_rep["Linhas"] / 1_000,
                     color=cores, edgecolor="white", linewidth=0.5, alpha=0.88)
        axes[0].set_title("Registros por Região (×1.000)", fontweight="bold")
        axes[0].set_xlabel("Registros (k)")
        axes[0].xaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{x:.0f}k"))
        axes[0].grid(axis="x", linestyle="--", alpha=0.4)
        sns.despine(ax=axes[0])
        for i, v in enumerate(df_rep["Linhas"]):
            axes[0].text(v/1_000 + 0.5, i, f"{v:,}", va="center", fontsize=8)

        axes[1].barh(df_rep["Regiao"], df_rep["Features_Add"],
                     color=cores, edgecolor="white", linewidth=0.5, alpha=0.88)
        axes[1].set_title("Features Adicionadas por Região", fontweight="bold")
        axes[1].set_xlabel("Nº de Features")
        axes[1].grid(axis="x", linestyle="--", alpha=0.4)
        sns.despine(ax=axes[1])
        for i, v in enumerate(df_rep["Features_Add"]):
            axes[1].text(v + 0.2, i, str(v), va="center", fontsize=8)

        fig.suptitle("Figura 4 — Resultado do Pipeline de Unificação por Região (SIN)",
                     fontweight="bold", y=1.02)
        plt.tight_layout()
    _salvar(fig, "FIG04_pipeline_por_regiao.png", dir_figuras)
    plt.show(); plt.close(fig)
    del df_rep; gc.collect()


def figura_variaveis_por_regiao_e_fonte(arquivo: str, dir_figuras: str) -> None:
    UI.step("Figura 5 — Variáveis por Região e por Fonte (Painel Duplo)")
    schema     = pq.read_schema(arquivo)
    all_cols   = schema.names
    cols_dados = [c for c in all_cols if c not in KEYS_TIME]

    contagem_regiao = {}
    for regiao in REGIOES_ORDEM:
        cols = [c for c in cols_dados if c.endswith(f"_{regiao}")]
        if cols:
            contagem_regiao[regiao] = len(cols)

    contagem_fonte = {"CCEE": 0, "INMET": 0, "BCB": 0, "ONS": 0}
    for col in cols_dados:
        u = col.upper()
        if "TARGET" in u:               contagem_fonte["CCEE"] += 1
        elif "METEO" in u:              contagem_fonte["INMET"] += 1
        elif "VALOR_COMPRA" in u or "VALOR_VENDA" in u:
                                        contagem_fonte["BCB"] += 1
        else:                           contagem_fonte["ONS"] += 1
    contagem_fonte = {k: v for k, v in contagem_fonte.items() if v > 0}
    total_vars = len(cols_dados)

    COR_FONTE = {"CCEE": "#9467bd", "INMET": "#17becf",
                 "BCB": "#e377c2",  "ONS":  "#8c564b"}

    def _label(n, v): return f"{n}  —  {v} vars ({v/total_vars*100:.1f}%)"
    def _central(ax, t):
        ax.text(0,  0.12, f"{t}", ha="center", va="center",
                fontsize=15, fontweight="bold")
        ax.text(0, -0.18, "variáveis\nno total", ha="center", va="center",
                fontsize=8.5, color="gray", linespacing=1.4)
    def _rosca(ax, vals, cores):
        wedges, _, ats = ax.pie(
            vals, labels=None, colors=cores, autopct="%1.1f%%",
            startangle=90, pctdistance=0.72,
            wedgeprops={"linewidth": 1.8, "edgecolor": "white", "width": 0.46},
        )
        for at in ats:
            at.set_fontsize(8.5); at.set_fontweight("bold"); at.set_color("white")
        return wedges

    with plt.rc_context(TCC_STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
        regioes   = list(contagem_regiao.keys())
        vals_reg  = list(contagem_regiao.values())
        cores_reg = [COR_REGIAO.get(r, "#7f7f7f") for r in regioes]
        w_reg = _rosca(axes[0], vals_reg, cores_reg)
        _central(axes[0], total_vars)
        axes[0].legend(w_reg, [_label(r, v) for r, v in zip(regioes, vals_reg)],
                       loc="lower center", bbox_to_anchor=(0.5, -0.18),
                       ncol=1, framealpha=0.85, fontsize=8.5, handlelength=1.2)
        axes[0].set_title("Por Região do SIN", fontweight="bold", pad=14)

        fontes    = list(contagem_fonte.keys())
        vals_font = list(contagem_fonte.values())
        cores_f   = [COR_FONTE.get(f, "#7f7f7f") for f in fontes]
        w_font = _rosca(axes[1], vals_font, cores_f)
        _central(axes[1], total_vars)
        axes[1].legend(w_font, [_label(f, v) for f, v in zip(fontes, vals_font)],
                       loc="lower center", bbox_to_anchor=(0.5, -0.18),
                       ncol=1, framealpha=0.85, fontsize=8.5, handlelength=1.2)
        axes[1].set_title("Por Fonte dos Dados", fontweight="bold", pad=14)

        fig.suptitle(
            "Figura 5 — Distribuição das Variáveis por Região do SIN e por Fonte de Dados",
            fontweight="bold", y=1.02)
        plt.tight_layout()
    _salvar(fig, "FIG05_rosca_variaveis_regiao_fonte.png", dir_figuras)
    plt.show(); plt.close(fig)
    gc.collect()


def tabela_resumo_pipeline(
    report_data, merge_metrics, t_total, arquivo_saida, dir_figuras
) -> pd.DataFrame:
    UI.step("Tabela 1 — Resumo do Pipeline (para TCC)")
    df_rep = pd.DataFrame(report_data)
    total  = {
        "Regiao": "TOTAL / BRASIL", "Status": "—",
        "Linhas": merge_metrics.get("total_linhas", "—"),
        "Features_Add": merge_metrics.get("total_colunas", "—"),
        "Memoria_MB":   merge_metrics.get("tamanho_mb", "—"),
        "Tempo_s":      round(t_total, 2),
    }
    df_rep["Memoria_MB"] = "—"; df_rep["Tempo_s"] = "—"
    df_tab = pd.concat([df_rep, pd.DataFrame([total])], ignore_index=True)[
        ["Regiao", "Status", "Linhas", "Features_Add", "Memoria_MB", "Tempo_s"]
    ]
    df_tab.columns = ["Região", "Status", "Registros",
                      "Features Adicionadas", "Memória (MB)", "Tempo (s)"]
    print(f"\n{df_tab.to_string(index=False)}\n")
    os.makedirs(dir_figuras, exist_ok=True)
    p = os.path.join(dir_figuras, "TAB01_resumo_pipeline.csv")
    df_tab.to_csv(p, index=False, encoding="utf-8-sig")
    UI.success(f"Tabela 1 salva: {p}")
    return df_tab


def tabela_auditoria_temporal(arquivo: str, dir_figuras: str) -> pd.DataFrame:
    UI.step("Tabela 2 — Auditoria Temporal por Ano")
    schema     = pq.read_schema(arquivo)
    cols_dados = [c for c in schema.names if c not in KEYS_TIME]
    anos       = sorted(pd.read_parquet(arquivo, columns=["KEY_ANO"])["KEY_ANO"].unique())

    registros = []
    CHUNK_COLS = 50
    for ano in anos:
        df_ano = pd.read_parquet(arquivo, columns=KEYS_TIME + cols_dados,
                                 filters=[("KEY_ANO", "=", ano)])
        meses = df_ano["KEY_MES"].nunique()
        n     = len(df_ano)
        soma, nc = 0.0, 0
        for i in range(0, len(cols_dados), CHUNK_COLS):
            bloco = cols_dados[i: i + CHUNK_COLS]
            soma += df_ano[bloco].isna().mean().mean()
            nc   += 1
        pct = (soma / nc) * 100 if nc else 0.0
        registros.append({
            "Ano": int(ano), "Registros": n, "Meses com Dados": meses,
            "% NaN Médio": round(pct, 2), "Completude (%)": round(100 - pct, 2),
        })
        del df_ano; gc.collect()

    df_tab = pd.DataFrame(registros)
    print(f"\n{df_tab.to_string(index=False)}\n")
    os.makedirs(dir_figuras, exist_ok=True)
    p = os.path.join(dir_figuras, "TAB02_auditoria_temporal.csv")
    df_tab.to_csv(p, index=False, encoding="utf-8-sig")
    UI.success(f"Tabela 2 salva: {p}")
    return df_tab


# ==============================================================================
# 🚀 EXECUÇÃO PRINCIPAL
# ==============================================================================

def main():
    UI.banner()
    t_start = time.time()

    for d in [DIR_SAIDA, DIR_FIGURAS, DIR_TMP]:
        os.makedirs(d, exist_ok=True)

    print(f" {UI.YELLOW}🔍 FILTRO DE ANOS ATIVO : {ANOS_FILTRO}{UI.RESET}")
    print(f" {UI.YELLOW}💾 Saída de figuras     : {DIR_FIGURAS}{UI.RESET}")
    print(f" {UI.YELLOW}📂 Dir temporário       : {DIR_TMP}{UI.RESET}\n")

    report_data = []
    paths_tmp   = []

    # ── FASE 1: salva cada região como parquet temporário (1 região na RAM) ────
    UI.step("FASE 1 — Preparação dos Parquets Temporários por Região")
    limpar_tmp(DIR_TMP)

    for regiao in REGIOES_ORDEM:
        UI.step(f"Processando Região: {regiao}")
        df_reg, r_rows, r_cols, ma, md = carregar_regiao(regiao)

        if df_reg is None:
            report_data.append({"Regiao": regiao, "Status": "VAZIO",
                                 "Linhas": 0, "Features_Add": 0})
            continue

        UI.info("Dimensões originais",    f"{r_rows:,} linhas × {r_cols} colunas")
        UI.info("Memória (antes/depois)", f"{ma:.1f} MB → {md:.1f} MB")

        path_tmp = os.path.join(DIR_TMP, f"TMP_{regiao}.parquet")
        df_reg.to_parquet(path_tmp, index=False, compression="snappy")
        paths_tmp.append(path_tmp)

        tam = os.path.getsize(path_tmp) / 1024**2
        UI.success(f"Parquet temporário salvo: {path_tmp} ({tam:.1f} MB)")

        report_data.append({
            "Regiao":       regiao,
            "Status":       "MERGED",
            "Linhas":       r_rows,
            "Features_Add": r_cols - len(KEYS_TIME),
        })

        del df_reg
        gc.collect()

    if not paths_tmp:
        UI.error("Nenhum dado foi preparado. Verifique os arquivos de entrada.")
        return

    # ── FASE 2: constrói schema unificado sem carregar dados ───────────────────
    schema_unificado = construir_schema_unificado(paths_tmp)

    # ── FASE 3: merge incremental com schema fixo ──────────────────────────────
    arquivo_saida = os.path.join(DIR_SAIDA, "DADOS_BRASIL_WIDE.parquet")
    merge_metrics = merge_parquets_por_chave(
        paths_tmp, arquivo_saida, schema_unificado, chunksize=5_000
    )

    if not merge_metrics:
        UI.error("Merge falhou.")
        return

    UI.info("Total de linhas",   f"{merge_metrics['total_linhas']:,}")
    UI.info("Total de colunas",  f"{merge_metrics['total_colunas']:,}")
    UI.info("Tamanho do arquivo", f"{merge_metrics['tamanho_mb']:.1f} MB")

    # ── FASE 4: auditoria de data de corte (lazy) ──────────────────────────────
    UI.step("Auditoria de Data de Corte — referência: METEO/INMET")
    corte = auditar_data_corte_meteo(arquivo_saida)

    print(f"\n{UI.MAGENTA}📡 STATUS DE ATUALIZAÇÃO DOS DADOS{UI.RESET}")
    if corte["ultimo_ano"] is not None:
        print(f"   ├─ Colunas METEO        : {UI.BOLD}{len(corte['cols_meteo'])}{UI.RESET}")
        print(f"   ├─ Último ano           : {UI.BOLD}{corte['ultimo_ano']}{UI.RESET}")
        print(f"   ├─ Último mês           : {UI.BOLD}{corte['ultimo_mes']:02d}{UI.RESET}")
        print(f"   ├─ Última data          : {UI.BOLD}{corte['ultima_data_fmt']}{UI.RESET}")
        print(f"   └─ Última hora          : {UI.BOLD}{corte['ultima_hora']:02d}:00{UI.RESET}")
    else:
        print(f"   └─ {UI.RED}Data de corte não determinada via METEO.{UI.RESET}")

    # ── FASE 5: figuras e tabelas (leitura lazy do parquet final) ─────────────
    t_total = time.time() - t_start

    figura_nan_por_ano(arquivo_saida,                  DIR_FIGURAS)
    figura_nan_por_regiao_ano(arquivo_saida,           DIR_FIGURAS)
    figura_pipeline_report(report_data,                DIR_FIGURAS)
    figura_variaveis_por_regiao_e_fonte(arquivo_saida, DIR_FIGURAS)

    tabela_resumo_pipeline(report_data, merge_metrics, t_total, arquivo_saida, DIR_FIGURAS)
    tabela_auditoria_temporal(arquivo_saida, DIR_FIGURAS)

    # ── FASE 6: limpeza ────────────────────────────────────────────────────────
    UI.step("Limpando arquivos temporários...")
    limpar_tmp(DIR_TMP)
    UI.success("Temporários removidos.")

    final_size = merge_metrics["tamanho_mb"]
    print(f"\n{UI.BLUE}╔{'═'*62}╗{UI.RESET}")
    print(f"{UI.BLUE}║{'RELATÓRIO FINAL DO PIPELINE':^62}║{UI.RESET}")
    print(f"{UI.BLUE}╚{'═'*62}╝{UI.RESET}")
    print(f"\n   📁 Arquivo gerado   : {os.path.basename(arquivo_saida)}")
    print(f"   💾 Tamanho em disco : {final_size:.2f} MB")
    print(f"   📊 Colunas totais   : {merge_metrics.get('total_colunas', '—')}")
    print(f"   📋 Linhas totais    : {merge_metrics.get('total_linhas', 0):,}")
    print(f"   ⏱️  Tempo total       : {t_total:.2f} segundos")
    print(f"   🖼️  Figuras geradas   : {DIR_FIGURAS}")
    if corte["ultimo_ano"]:
        print(f"   📅 Corte METEO       : {corte['ultima_data_fmt']} {corte['ultima_hora']:02d}:00h")
    print(f"\n{UI.GREEN}✅ PIPELINE CONCLUÍDO COM SUCESSO!{UI.RESET}\n")


if __name__ == "__main__":
    main()